# Course Recommender — Training on Udemy Courses (hossaingh/udemy-courses)

Trains **TF-IDF + TruncatedSVD** (fit by you, not a pretrained model) on `Course_info.csv`
(~210k distinct Udemy courses — real courses, not review-duplicated rows).

KNN is NOT persisted here — it's fit fresh on the vector pool at query time in the
runtime app (`data_store.py`), so it always reflects added/deleted entries.

## Before you run
1. Create a Kaggle Notebook.
2. **Add Input -> Datasets** -> attach `hossaingh/udemy-courses`.
3. Run all cells. Outputs land in `/kaggle/working/`:
   `courses.csv`, `tfidf.pkl`, `svd.pkl`, `vectors.npy`, `report.json`.
4. Download those and drop them into the PBEL project root.

## 1. Setup

In [1]:
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors

OUT_DIR = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
MIN_ROWS = 100_000  # require > 1 lakh rows
N_SVD_COMPONENTS = 200
MAX_TFIDF_FEATURES = 5000
RANDOM_STATE = 42

assert INPUT_ROOT.exists(), "/kaggle/input not found — are you running this outside Kaggle?"

## 2. Load Course_info.csv

Finds the real course-metadata file (NOT `Comments.csv` — that's 9M+ user comments,
and using it as "courses" would just recreate the review-duplication problem).

In [2]:
print("Files under /kaggle/input:")
for p in sorted(INPUT_ROOT.rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size / 1e6:.2f} MB)")

def find_course_info() -> Path:
    hits = [p for p in INPUT_ROOT.rglob("*.csv") if "course_info" in p.name.lower()]
    if not hits:
        for p in INPUT_ROOT.rglob("*.csv"):
            if "comment" in p.name.lower():
                continue
            cols = pd.read_csv(p, nrows=0).columns.str.lower().tolist()
            if any("title" in c for c in cols):
                hits.append(p)
    assert hits, (
        "Could not find Course_info.csv under /kaggle/input. "
        "Make sure the hossaingh/udemy-courses dataset is attached."
    )
    return hits[0]

course_info_path = find_course_info()
print("\nUsing:", course_info_path)

raw = pd.read_csv(course_info_path)
print("\nShape:", raw.shape)
print("Columns:", raw.columns.tolist())
raw.head(3)

Files under /kaggle/input:
  /kaggle/input/datasets/hossaingh/udemy-courses/Comments.csv  (1609.43 MB)
  /kaggle/input/datasets/hossaingh/udemy-courses/Course_info.csv  (76.15 MB)

Using: /kaggle/input/datasets/hossaingh/udemy-courses/Course_info.csv

Shape: (209734, 20)
Columns: ['id', 'title', 'is_paid', 'price', 'headline', 'num_subscribers', 'avg_rating', 'num_reviews', 'num_comments', 'num_lectures', 'content_length_min', 'published_time', 'last_update_date', 'category', 'subcategory', 'topic', 'language', 'course_url', 'instructor_name', 'instructor_url']


,id,title,is_paid,price,headline,num_subscribers,avg_rating,num_reviews,num_comments,num_lectures,content_length_min,published_time,last_update_date,category,subcategory,topic,language,course_url,instructor_name,instructor_url
0,4715.0,Online Vegan Vegetarian Cooking School,True,24.99,Learn to cook delicious vegan recipes. Filmed ...,2231.0,3.75,134.0,42.0,37.0,1268.0,2010-08-05T22:06:13Z,2020-11-06,Lifestyle,Food & Beverage,Vegan Cooking,English,/course/vegan-vegetarian-cooking-school/,Angela Poch,/user/angelapoch/
1,1769.0,The Lean Startup Talk at Stanford E-Corner,False,0.00,Debunking Myths of Entrepreneurship A startup ...,26474.0,4.50,709.0,112.0,9.0,88.0,2010-01-12T18:09:46Z,NaN,Business,Entrepreneurship,Lean Startup,English,/course/the-lean-startup-debunking-myths-of-en...,Eric Ries,/user/ericries/
2,5664.0,"How To Become a Vegan, Vegetarian, or Flexitarian",True,19.99,Get the tools you need for a lifestyle change ...,1713.0,4.40,41.0,13.0,14.0,82.0,2010-10-13T18:07:17Z,2019-10-09,Lifestyle,Other Lifestyle,Vegan Cooking,English,/course/see-my-personal-motivation-for-becomin...,Angela Poch,/user/angelapoch/


## 3. Verify row count and inspect available text columns

Fails loudly instead of silently training on too little data or no text signal.

In [3]:
assert len(raw) >= MIN_ROWS, (
    f"Only {len(raw):,} rows found — need >= {MIN_ROWS:,}. "
    "Wrong file, or dataset version changed."
)
print(f"Row count OK: {len(raw):,} >= {MIN_ROWS:,}")

def _pick(df, names):
    lower = {c.lower().strip(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None

id_col = _pick(raw, ["id", "course_id", "courseId"])
title_col = _pick(raw, ["title", "course_title"])
headline_col = _pick(raw, ["headline", "subtitle"])
category_col = _pick(raw, ["category", "subject", "subcategory"])
language_col = _pick(raw, ["language", "lang"])
is_paid_col = _pick(raw, ["is_paid"])
price_col = _pick(raw, ["price"])
subs_col = _pick(raw, ["num_subscribers", "subscribers"])
rating_col = _pick(raw, ["avg_rating", "rating"])

assert id_col is not None, "No id column found."
assert title_col is not None, "No title column found — cannot build course text."

print("id_col:", id_col)
print("title_col:", title_col)
print("headline_col:", headline_col)
print("category_col:", category_col)
print("language_col:", language_col)

if headline_col is None and category_col is None:
    print(
        "\nWARNING: only 'title' is available as text. "
        "TF-IDF will be thin (short strings) — search quality will lean heavily "
        "on keyword overlap in titles rather than rich semantic content. "
        "This is a known tradeoff of this dataset vs. MOOCCube's longer descriptions."
    )

Row count OK: 209,734 >= 100,000
id_col: id
title_col: title
headline_col: headline
category_col: category
language_col: language


## 4. Clean, filter to English (if a language column exists), dedupe

In [4]:
df = raw.copy()

before = len(df)
if language_col is not None:
    langs = df[language_col].astype(str).str.lower()
    english_mask = langs.str.contains("english") | (langs == "en")
    if english_mask.sum() >= MIN_ROWS:
        df = df[english_mask].copy()
        print(f"Filtered to English rows: {before:,} -> {len(df):,}")
    else:
        print(
            f"English-only filter would drop below {MIN_ROWS:,} rows "
            f"({english_mask.sum():,} would remain) — keeping all languages instead. "
            "Note: mixed-language TF-IDF vocabulary will be noisier."
        )
else:
    print("No language column found — keeping all rows (dataset may be multilingual).")

df[title_col] = df[title_col].fillna("").astype(str).str.strip()
df = df[df[title_col] != ""]

before_dedupe = len(df)
df = df.drop_duplicates(subset=[title_col]).reset_index(drop=True)
print(f"Dropped {before_dedupe - len(df):,} duplicate-title rows -> {len(df):,} remain")

assert len(df) >= MIN_ROWS, (
    f"After cleaning/dedupe only {len(df):,} rows remain — below the {MIN_ROWS:,} requirement. "
    "Loosen the language filter or dedupe key."
)

Filtered to English rows: 209,734 -> 123,921
Dropped 994 duplicate-title rows -> 122,927 remain


## 5. Build the content field used for TF-IDF

Uses every real text/category signal available — title is mandatory, headline and
category are folded in when present to reduce thinness.

In [5]:
parts = [df[title_col]]
if headline_col is not None:
    parts.append(df[headline_col].fillna("").astype(str))
if category_col is not None:
    parts.append(df[category_col].fillna("").astype(str))

content = parts[0]
for p in parts[1:]:
    content = content + " " + p

df["content"] = content.str.strip()

avg_len = df["content"].str.len().mean()
print(f"Avg content length: {avg_len:.0f} chars")
print("\nSample content values:")
for s in df["content"].head(5):
    print(" -", s[:120])

Avg content length: 142 chars

Sample content values:
 - Online Vegan Vegetarian Cooking School Learn to cook delicious vegan recipes. Filmed over 15 years ago, watch the first 
 - The Lean Startup Talk at Stanford E-Corner Debunking Myths of Entrepreneurship A startup is not a "doll house" version o
 - How To Become a Vegan, Vegetarian, or Flexitarian Get the tools you need for a lifestyle change that will bring you heal
 - How to Train a Puppy Train your puppy the right way with Dr. Ian Dunbar. Includes 13 videos, 4 books, and 16 behavior bl
 - Web Design from the Ground Up Learn web design online: Everything you need to know about XHTML and CSS, the basic buildi


## 6. Build the runtime schema (course_id, course_name, course_about, popularity_score, enrollment_count)

Matches columns the PBEL Streamlit/Flask app already expects — no downstream code changes.

In [6]:
courses = pd.DataFrame({
    "course_id": df[id_col].astype(str).values,
    "course_name": df[title_col].values,
    "course_about": (
        df[headline_col].fillna("").astype(str).values
        if headline_col is not None
        else [""] * len(df)
    ),
})

if subs_col is not None:
    subs = df[subs_col].fillna(0).astype(float)
    max_subs = subs.max() if subs.max() > 0 else 1.0
    courses["popularity_score"] = (subs / max_subs).round(4).values
    courses["enrollment_count"] = df[subs_col].fillna(0).astype(int).values
else:
    courses["popularity_score"] = 0.0
    courses["enrollment_count"] = 0

courses.insert(0, "id", [f"seed-{cid}" for cid in courses["course_id"]])

courses.head(3)

,id,course_id,course_name,course_about,popularity_score,enrollment_count
0,seed-4715.0,4715.0,Online Vegan Vegetarian Cooking School,Learn to cook delicious vegan recipes. Filmed ...,0.0013,2231
1,seed-1769.0,1769.0,The Lean Startup Talk at Stanford E-Corner,Debunking Myths of Entrepreneurship A startup ...,0.0151,26474
2,seed-5664.0,5664.0,"How To Become a Vegan, Vegetarian, or Flexitarian",Get the tools you need for a lifestyle change ...,0.0010,1713


## 7. Fit TF-IDF + TruncatedSVD (this is the actual training step)

In [7]:
tfidf = TfidfVectorizer(stop_words="english", max_features=MAX_TFIDF_FEATURES, min_df=2, max_df=0.8)
tfidf_matrix = tfidf.fit_transform(df["content"])
print("TF-IDF matrix:", tfidf_matrix.shape)

svd = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=RANDOM_STATE)
vectors = svd.fit_transform(tfidf_matrix).astype(np.float32)
print("Reduced vectors:", vectors.shape)
print(f"Explained variance (SVD, {N_SVD_COMPONENTS} components): {svd.explained_variance_ratio_.sum():.3f}")

TF-IDF matrix: (122927, 5000)
Reduced vectors: (122927, 200)
Explained variance (SVD, 200 components): 0.314


## 8. Quick sanity check — qualitative search

In [8]:
def recommend(query, k=5):
    q_vec = tfidf.transform([query])
    if q_vec.nnz == 0:
        print(f"'{query}' -> no vocabulary overlap (OOV), no results")
        return
    q_reduced = svd.transform(q_vec)
    knn = NearestNeighbors(n_neighbors=k, metric="cosine")
    knn.fit(vectors)
    dist, idx = knn.kneighbors(q_reduced)
    for d, i in zip(dist[0], idx[0]):
        print(f"  {1-d:.3f}  {courses.iloc[i]['course_name']}")

for q in ["python machine learning", "excel for beginners", "guitar lessons"]:
    print(f"\nQuery: {q}")
    recommend(q, k=5)


Query: python machine learning
  0.964  Machine Learning Top 5 Models Implementation "A-Z"
  0.961  Practical Machine Learning with Python
  0.955  Clustering & Classification With Machine Learning In Python
  0.955  Machine Learning with Python
  0.954  Boosting Machine Learning Models in Python

Query: excel for beginners
  0.873  KNIME for Microsoft Excel Users
  0.871  Microsoft Excel - Excel Only For Beginners 2022
  0.847  Microsoft Excel - Excel for beginners
  0.838  A beginners guide to Excel
  0.832  Most Essential & Popular Excel Formulas And Functions - 2022

Query: guitar lessons
  0.981  Blues Guitar Lessons - From Texas To Carolina
  0.980  Blues Guitar Lessons - Robert Johnson and Scrapper Blackwell
  0.963  Acoustic Blues Guitar Lessons : Learn Acoustic Blues Guitar
  0.960  Blues Guitar Lessons - Ragtime Blues Guitar
  0.959  Fingerstyle Guitar Lessons : Blues Chords : Fingerpicking


## 9. Export artifacts for PBEL

Download `courses.csv`, `tfidf.pkl`, `svd.pkl`, `vectors.npy` from the Kaggle Output
panel and place them in the PBEL project root (replacing the old MOOCCube-trained files).

No `knn_model.pkl` is exported — `data_store.py` fits KNN fresh on `vectors.npy`
at query time so it always reflects current add/delete state.

In [9]:
courses.to_csv(OUT_DIR / "courses.csv", index=False)
joblib.dump(tfidf, OUT_DIR / "tfidf.pkl")
joblib.dump(svd, OUT_DIR / "svd.pkl")
np.save(OUT_DIR / "vectors.npy", vectors)

report = {
    "source_file": str(course_info_path),
    "rows_after_cleaning": len(courses),
    "min_rows_required": MIN_ROWS,
    "tfidf_features": tfidf_matrix.shape[1],
    "svd_components": N_SVD_COMPONENTS,
    "svd_explained_variance": float(svd.explained_variance_ratio_.sum()),
    "content_fields_used": [c for c in [title_col, headline_col, category_col] if c],
    "language_filtered": language_col is not None,
}
with open(OUT_DIR / "report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nSaved to:", OUT_DIR)
for p in ["courses.csv", "tfidf.pkl", "svd.pkl", "vectors.npy", "report.json"]:
    print(" -", OUT_DIR / p)

{
  "source_file": "/kaggle/input/datasets/hossaingh/udemy-courses/Course_info.csv",
  "rows_after_cleaning": 122927,
  "min_rows_required": 100000,
  "tfidf_features": 5000,
  "svd_components": 200,
  "svd_explained_variance": 0.31352634101970367,
  "content_fields_used": [
    "title",
    "headline",
    "category"
  ],
  "language_filtered": true
}

Saved to: /kaggle/working
 - /kaggle/working/courses.csv
 - /kaggle/working/tfidf.pkl
 - /kaggle/working/svd.pkl
 - /kaggle/working/vectors.npy
 - /kaggle/working/report.json
